# Complexity-02 — Vérifier ou trouver : P, NP et la réduction

**Prérequis** : notions de tableaux, boucles et fonctions en C#; aucun package externe. Toutes les expériences utilisent la bibliothèque standard.

**Navigation** : [Version Python](Complexity-02-P-NP-Reduction-Python.ipynb) · [<< Complexity-01 — Compter des pas](Complexity-01-StepCounting-Python.ipynb) | [Index de la série](README.md) | [Complexity-02b — Le seuil du 3-SAT, mesuré contre la densité >>](Complexity-02b-SAT-PhaseTransition-Python.ipynb) | [Complexity-03 — Plus de temps, plus de problèmes >>](Complexity-03-HierarchieTemporelle-Python.ipynb) | [02c — approfondissement : seuil sans mur, GF(2)](Complexity-02c-XOR-SAT-Phase-Transition-Python.ipynb)

> **Public : Découverte.** Deuxième notebook de la série `Complexity/`. Il prolonge directement
> le premier et ne suppose rien d'autre.

## Ce que ce notebook suppose

- [Complexity-01 — Compter des pas](Complexity-01-StepCounting-Python.ipynb) : compter les **pas** d'un
  algorithme, la **taille** $n$ d'une entrée, le **pire cas**, le **test du doublement**, et la
  frontière entre coût **polynomial** ($n$, $n^2$, $n^3$…) et coût **exponentiel** ($2^n$) ;
- lire un programme C# simple (boucles, fonctions récursives, listes, dictionnaires).

## Ce que vous saurez faire à la fin

1. distinguer **trouver** une solution et **vérifier** une solution proposée — le **certificat** ;
2. définir les classes **P** et **NP**, et dire ce que NP ne veut **pas** dire ;
3. reconnaître le piège du coût **pseudo-polynomial** ;
4. exécuter une **réduction** de bout en bout : transformer une instance, la résoudre, retraduire
   la solution, la vérifier ;
5. expliquer pourquoi l'exponentielle reste un mur même pour une machine mille fois plus rapide.

**Durée estimée** : 60 minutes.

In [1]:
// Bibliothèque standard uniquement : solveurs et vérificateurs sont écrits dans ce carnet.
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Linq;
using System.Numerics;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;
Console.WriteLine($"C# {Environment.Version} · culture numérique invariante · bibliothèque standard uniquement.");

C# 10.0.14 · culture numérique invariante · bibliothèque standard uniquement.


## 1. Le Sudoku : trouver coûte, vérifier ne coûte presque rien

Un Sudoku illustre l'écart mieux qu'un long discours. **Trouver** la grille complète demande de la
recherche : le solveur le plus simple, le **backtracking** (retour arrière), remplit la première
case vide avec le premier chiffre compatible, continue, et revient en arrière dès qu'il est bloqué.
**Vérifier** une grille proposée demande seulement de relire ses 27 unités (9 rangées, 9 colonnes,
9 blocs) et ses indices de départ.

Le solveur ci-dessous est la version minimale de celui de
[Sudoku-01 — Backtracking](../Sudoku/Sudoku-01-Backtracking-CSharp.ipynb) ; il compte ses
**appels récursifs**, le vérificateur compte ses **lectures de cases**. Les quatre grilles viennent
des fichiers de puzzles de la série Sudoku (`Sudoku_Easy51.txt` et `Sudoku_hardest.txt`).

In [2]:
var grilles = new Dictionary<string, string>
{
    ["facile A"] = "902005403100063025508407060026309001057010290090670530240530600705200304080041950",
    ["facile B"] = "000000907000420180000705026100904000050000040000507009920108000034059000507000000",
    ["difficile A"] = "85...24..72......9..4.........1.7..23.5...9...4...........8..7..17..........36.4.",
    ["difficile B"] = "12..4......5.69.1...9...5.........7.7...52.9..3......2.9.6...5.4..9..8.1..3...9.4"
};
int[] LireGrille(string texte) => texte.Select(c => c == '.' || c == '0' ? 0 : c - '0').ToArray();
bool Compatible(int[] g, int k, int valeur)
{
    int ligne = k / 9, colonne = k % 9;
    for (int j = 0; j < 9; j++)
    {
        if (g[ligne * 9 + j] == valeur || g[j * 9 + colonne] == valeur) return false;
        int bloc = (ligne / 3 * 3 + j / 3) * 9 + colonne / 3 * 3 + j % 3;
        if (g[bloc] == valeur) return false;
    }
    return true;
}
long Resoudre(int[] g)
{
    long appels = 0;
    bool Chercher()
    {
        appels++;
        int k = Array.IndexOf(g, 0);
        if (k < 0) return true;
        for (int v = 1; v <= 9; v++)
        {
            if (!Compatible(g, k, v)) continue;
            g[k] = v;
            if (Chercher()) return true;
            g[k] = 0;
        }
        return false;
    }
    Chercher();
    return appels;
}
(int r, int c)[][] Unites()
{
    var u = new List<(int r, int c)[]>();
    for (int r = 0; r < 9; r++) u.Add(Enumerable.Range(0, 9).Select(c => (r,c)).ToArray());
    for (int c = 0; c < 9; c++) u.Add(Enumerable.Range(0, 9).Select(r => (r,c)).ToArray());
    for (int br = 0; br < 3; br++) for (int bc = 0; bc < 3; bc++)
        u.Add((from dr in Enumerable.Range(0,3) from dc in Enumerable.Range(0,3) select (br*3+dr,bc*3+dc)).ToArray());
    return u.ToArray();
}
(int Accepte, int Lectures) VerifierSudoku(int[] depart, int[] proposee)
{
    int lectures = 0;
    for (int i = 0; i < 81; i++) { lectures++; if (depart[i] != 0 && proposee[i] != depart[i]) return (0,lectures); }
    foreach (var unite in Unites())
    {
        var vus = new HashSet<int>();
        foreach (var (r,c) in unite) { lectures++; vus.Add(proposee[r*9+c]); }
        if (vus.Count != 9 || vus.Any(v => v < 1 || v > 9)) return (0,lectures);
    }
    return (1,lectures);
}
Console.WriteLine("Grille       | appels de recherche | lectures vérificateur | trouver / vérifier");
var solutions = new Dictionary<string,int[]>();
foreach (var (nom, texte) in grilles)
{
    var depart = LireGrille(texte); var proposee = (int[])depart.Clone();
    long appels = Resoudre(proposee); var (ok, lectures) = VerifierSudoku(depart,proposee);
    if (ok != 1 || lectures != 324) throw new Exception($"La solution ou le compte Sudoku est invalide: {nom}.");
    solutions[nom] = proposee;
    double rapport = (double)appels/lectures;
    string rapportTxt = rapport < 10 ? rapport.ToString("F2",CultureInfo.InvariantCulture) : Math.Round(rapport).ToString("N0",CultureInfo.InvariantCulture);
    Console.WriteLine($"{nom,-12} | {appels,19:N0} | {lectures,21:N0} | {rapportTxt}×");
}
var departFacile = LireGrille(grilles["facile A"]);
var falsifie = (int[])solutions["facile A"].Clone();
int k1 = -1, k2 = -1;
for (int a = 0; a < 81 && k1 < 0; a++) for (int b = a+1; b < 81; b++)
    if (a/9 == b/9 && departFacile[a] == 0 && departFacile[b] == 0) { k1=a; k2=b; break; }
(falsifie[k1],falsifie[k2]) = (falsifie[k2],falsifie[k1]);
var rejet = VerifierSudoku(departFacile,falsifie);
if (rejet.Accepte != 0) throw new Exception("Le faux certificat Sudoku a été accepté.");
Console.WriteLine($"Faux certificat échangé : rejeté après {rejet.Lectures} lectures.");

Grille       | appels de recherche | lectures vérificateur | trouver / vérifier


facile A     |                  49 |                   324 | 0.15×


facile B     |              19,023 |                   324 | 59×


difficile A  |             335,638 |                   324 | 1,036×


difficile B  |             228,215 |                   324 | 704×


Faux certificat échangé : rejeté après 180 lectures.


**Lecture.** Les quatre grilles exactes demandent respectivement **49**, **19 023**, **335 638** et **228 215** appels, tandis que le vérificateur effectue toujours **324 lectures**. Le rapport atteint **1 036×** pour la grille difficile A. Le certificat échangé est rejeté après **180 lectures** : vérifier recalcule les contraintes et compare les indices de départ.

Pour un Sudoku généralisé $m^2 \times m^2$, vérifier demande au plus $4m^4$ lectures, linéaires en nombre de cases. Le backtracking peut explorer un nombre exponentiel de branches. L'expérience mesure le coût sur ces quatre puzzles; la NP-complétude du Sudoku généralisé est citée (Yato & Seta, 2003).

## Exercices

Les trois cellules d'exercice restent volontairement non résolues. Remplacez le `return null` par une implémentation; les gardes n'affichent un résultat que si les fixtures, dimensions, certificats et limites de coût sont respectés. Les exercices apparaissent dans les sections où les notions sont introduites.

### Exercice 1 — Certificat de coloriage (Petersen)

Le graphe de Petersen contient exactement 10 sommets numérotés `0..9` et 15 arêtes. Deux tableaux de 10 couleurs sont fournis pour `k=3`: le premier est un coloriage valide, le second contient un conflit. Implémentez `VerifierColoriage(aretes, couleurs, k)`, qui renvoie `(accepte, operations)` ou `null` tant qu'il n'est pas terminé. Il faut vérifier la taille exacte, la plage des couleurs et les extrémités de chaque arête, puis s'arrêter au premier conflit. Le contrôle attend l'acceptation de A après les 15 arêtes, le rejet de B, et le rejet préalable d'un certificat tronqué ou avec une couleur hors de `0..2`. Une entrée mal formée doit compter zéro arête examinée.

In [3]:
var petersen = new (int A,int B)[]
{
    (0,1),(1,2),(2,3),(3,4),(4,0), (0,5),(1,6),(2,7),(3,8),(4,9),
    (5,7),(7,9),(9,6),(6,8),(8,5)
};
int[] certificatA = { 0,1,0,1,2,1,0,2,2,1 };
int[] certificatB = { 0,1,0,1,2,1,0,2,2,0 };
(int Accepte,int Operations)? VerifierColoriage((int A,int B)[] aretes,int[] couleurs,int k)
{
    // TODO étudiant : vérifier les bornes, puis chaque arête; compter les arêtes examinées.
    return null;
}
bool dimensionsExactes = petersen.Length == 15 && certificatA.Length == 10 && certificatB.Length == 10 && petersen.All(e => e.A >= 0 && e.A < 10 && e.B >= 0 && e.B < 10);
bool couleursDansDomaine = certificatA.Concat(certificatB).All(c => c >= 0 && c < 3);
var testA = dimensionsExactes && couleursDansDomaine ? VerifierColoriage(petersen,certificatA,3) : null;
var testB = dimensionsExactes && couleursDansDomaine ? VerifierColoriage(petersen,certificatB,3) : null;
var horsDomaine = (int[])certificatA.Clone(); horsDomaine[0] = 3;
var tailleInvalide = certificatA.Take(9).ToArray();
var rejetCouleur = dimensionsExactes ? VerifierColoriage(petersen,horsDomaine,3) : null;
var rejetTaille = dimensionsExactes ? VerifierColoriage(petersen,tailleInvalide,3) : null;
bool malformesRejetes = rejetCouleur.HasValue && rejetCouleur.Value.Accepte == 0 && rejetCouleur.Value.Operations == 0 &&
    rejetTaille.HasValue && rejetTaille.Value.Accepte == 0 && rejetTaille.Value.Operations == 0;
if (testA.HasValue && testB.HasValue && testA.Value.Accepte == 1 && testA.Value.Operations == 15 &&
    testB.Value.Accepte == 0 && testB.Value.Operations >= 1 && testB.Value.Operations <= 15 && malformesRejetes)
    Console.WriteLine($"Petersen: A accepté ({testA.Value.Operations}/15 arêtes), B rejeté ({testB.Value.Operations}/15); taille et couleur invalides rejetées avant parcours.");
else Console.WriteLine("Exercice non résolu : implémentez VerifierColoriage; aucun verdict ni compte n'est accepté en stub.");

Exercice non résolu : implémentez VerifierColoriage; aucun verdict ni compte n'est accepté en stub.


**Lecture.** Le stub ne calcule aucun verdict et n'affiche aucun faux compte. Une réponse complète doit accepter A après les 15 arêtes et détecter le conflit de B dans la limite de 15; les deux certificats ont exactement 10 couleurs dans `{0,1,2}`.

## 2. Subset-sum : le certificat, et le piège du pseudo-polynomial

Le problème **subset-sum** (somme de sous-ensemble) est le plus simple des problèmes difficiles :

> **Instance** : une liste de poids entiers positifs et une cible $t$.
> **Question** : existe-t-il un sous-ensemble des poids dont la somme vaut exactement $t$ ?

C'est un **problème de décision** : la réponse est *oui* ou *non*. Quand elle est *oui*, un
**certificat** la prouve : la liste des indices des poids choisis. Le vérifier coûte une addition
par indice.

Pour **trouver**, deux algorithmes :

- la **force brute** essaie les $2^n$ sous-ensembles ;
- la **programmation dynamique** remplit une table des sommes atteignables $0, 1, \dots, t$.

In [4]:
(bool Verdict, long Operations, int[] Certificat) SubsetBrut(int[] poids, int cible)
{
    long operations = 0;
    int limite = 1 << poids.Length;
    for (int masque = 0; masque < limite; masque++)
    {
        int somme = 0; var indices = new List<int>();
        for (int i = 0; i < poids.Length; i++)
        {
            operations++;
            if ((masque & (1 << i)) != 0) { somme += poids[i]; indices.Add(i); }
        }
        if (somme == cible) return (true, operations, indices.ToArray());
    }
    return (false, operations, null);
}
(bool Verdict, long Operations) SubsetDp(int[] poids, int cible)
{
    if (cible < 0) return (false, 0);
    var dp = new bool[cible + 1]; dp[0] = true; long operations = 0;
    foreach (int p in poids)
        for (int s = cible; s >= p; s--) { operations++; if (dp[s-p]) dp[s] = true; }
    return (dp[cible], operations);
}
(bool Accepte, int Operations) VerifierSubset(int[] poids, int cible, int[] indices)
{
    int operations = 0; var vus = new HashSet<int>(); int somme = 0;
    foreach (int i in indices)
    {
        operations++;
        if (i < 0 || i >= poids.Length || !vus.Add(i)) return (false, operations);
        somme += poids[i];
    }
    return (somme == cible, operations);
}
var rng = new Random(42);
Console.WriteLine(" n | verdict | opérations brut | opérations DP | vérification certificat");
foreach (int n in new[] { 8, 11, 14, 17 })
{
    var poids = Enumerable.Range(0,n).Select(_ => rng.Next(1,60)).ToArray();
    int cible = poids.Sum() / 2;
    var brut = SubsetBrut(poids,cible); var dp = SubsetDp(poids,cible);
    if (brut.Verdict != dp.Verdict) throw new Exception("Brut et DP divergent.");
    string verif = "—";
    if (brut.Verdict)
    {
        var v = VerifierSubset(poids,cible,brut.Certificat);
        if (!v.Accepte) throw new Exception("Certificat subset-sum invalide.");
        verif = v.Operations.ToString("N0",CultureInfo.InvariantCulture);
    }
    else if (brut.Certificat != null) throw new Exception("Un certificat inattendu accompagne un non.");
    Console.WriteLine($"{n,2} | {(brut.Verdict ? "oui" : "non"),7} | {brut.Operations,15:N0} | {dp.Operations,13:N0} | {verif,22}");
}

 n | verdict | opérations brut | opérations DP | vérification certificat


 8 |     non |           2,048 |           572 |                      —


11 |     oui |           1,045 |         1,162 |                      5


14 |     oui |           3,472 |         2,342 |                      7


17 |     oui |          16,983 |         3,182 |                      7


**Lecture.** Les quatre instances rendent le même verdict par force brute et DP. À $n=8$, le non parcourt exactement **2 048** combinaisons-poids et la DP examine **572** transitions. Pour les trois réponses oui, les certificats sont vérifiés en **5**, **7** et **7** opérations. La force brute s'arrête au premier sous-ensemble valide; ce tableau compare ces instances déterministes, pas une loi de domination sur toutes les entrées.

In [5]:
Console.WriteLine(" n | opérations brut (sans solution) | n × 2^n | facteur pour 2 poids de plus");
long? precedent = null;
foreach (int n in new[] { 10, 12, 14, 16, 18 })
{
    var poids = Enumerable.Repeat(2,n).ToArray();
    var brut = SubsetBrut(poids, 2*n-1);
    long exact = (long)n * (1L << n);
    if (brut.Verdict || brut.Operations != exact) throw new Exception("Le cas impossible ne réalise pas exactement n × 2^n.");
    string facteur = precedent.HasValue ? (brut.Operations/(double)precedent.Value).ToString("F2", CultureInfo.InvariantCulture) : "—";
    Console.WriteLine($"{n,2} | {brut.Operations,29:N0} | {exact,9:N0} | {facteur}");
    precedent = brut.Operations;
}

 n | opérations brut (sans solution) | n × 2^n | facteur pour 2 poids de plus


10 |                        10,240 |    10,240 | —


12 |                        49,152 |    49,152 | 4.80


14 |                       229,376 |   229,376 | 4.67


16 |                     1,048,576 | 1,048,576 | 4.57


18 |                     4,718,592 | 4,718,592 | 4.50


**Lecture.** Dans les cas sans solution, les compteurs vérifient exactement $n \cdot 2^n$: de **10 240** opérations à $n=10$ jusqu'à **4 718 592** à $n=18$. Le facteur lorsqu'on ajoute deux poids passe de **4,80** à **4,50**, conformément à la formule $n \cdot 2^n$.

In [6]:
var rngEchelle = new Random(7);
int[] basePoids = Enumerable.Range(0,14).Select(_ => rngEchelle.Next(1,60)).ToArray();
int cibleBase = basePoids.Sum()/2;
Console.WriteLine("facteur | chiffres entrée | verdict | brut opérations | DP opérations | nombre d'états DP");
foreach (int facteur in new[] { 1, 10, 100, 1000, 10000 })
{
    int[] poids = basePoids.Select(p => checked(p*facteur)).ToArray();
    int cible = checked(cibleBase*facteur);
    var brut = SubsetBrut(poids,cible); var dp = SubsetDp(poids,cible);
    if (brut.Verdict != dp.Verdict) throw new Exception("Le changement d'échelle a modifié le verdict.");
    int chiffres = poids.Append(cible).Sum(v => v.ToString(CultureInfo.InvariantCulture).Length);
    Console.WriteLine($"{facteur,7:N0} | {chiffres,14} | {(brut.Verdict ? "oui" : "non"),7} | {brut.Operations,15:N0} | {dp.Operations,13:N0} | {cible+1,15:N0}");
}

facteur | chiffres entrée | verdict | brut opérations | DP opérations | nombre d'états DP


      1 |             28 |     oui |           5,264 |         2,762 |             230


     10 |             43 |     oui |           5,264 |        27,494 |           2,291


    100 |             58 |     oui |           5,264 |       274,814 |          22,901


  1,000 |             73 |     oui |           5,264 |     2,748,014 |         229,001


 10,000 |             88 |     oui |           5,264 |    27,480,014 |       2,290,001


**Lecture.** Pour l'instance générée par `Random(7)`, les cinq facteurs gardent le même verdict oui et la force brute examine **5 264** poids candidats : elle ne dépend que des combinaisons. La DP passe de **2 762** à **27 480 014** transitions lorsque les poids sont multipliés par 10 000. La longueur totale écrite des 14 poids et de la cible passe de **28 à 88 chiffres**. Le coût est polynomial en la valeur de la cible, mais exponentiel en sa longueur d'écriture : c'est un algorithme pseudo-polynomial. La séquence .NET diffère de celle de NumPy, mais le seed, les 14 poids et les cinq facteurs sont conservés.

## 3. P et NP

Les deux sections précédentes montrent le même motif : **vérifier** un certificat est facile,
**trouver** une solution semble difficile. La théorie de la complexité donne un nom à chaque côté.

| Classe | Définition | Exemples |
|---|---|---|
| **P** | les problèmes de décision qu'un algorithme **résout** en un nombre de pas polynomial en la taille de l'entrée | trier puis chercher, tester si un nombre est premier, plus court chemin |
| **NP** | les problèmes de décision dont chaque instance *oui* possède un **certificat** de taille polynomiale, **vérifiable** en un nombre de pas polynomial | Sudoku généralisé, subset-sum, coloriage de graphe en 3 couleurs |

Trois remarques essentielles :

1. **NP ne veut pas dire « non polynomial ».** Le N vient de *non déterministe* : une machine qui
   pourrait **deviner** le bon certificat n'aurait plus qu'à le vérifier. NP est la classe du
   « facile à vérifier », pas celle du « difficile ».
2. **P est inclus dans NP** : si l'on sait trouver la réponse en temps polynomial, on sait la
   vérifier — il suffit d'ignorer le certificat et de recalculer.
3. **La question P = NP est ouverte.** Personne ne sait s'il existe un problème de NP qui ne soit
   pas dans P. C'est l'un des sept *problèmes du millénaire* du Clay Mathematics Institute, posé
   formellement par Stephen Cook en 1971.

Pour avancer malgré cette ignorance, on **compare** les problèmes entre eux. C'est l'outil de la
section suivante.

## 4. Une réduction exécutée de bout en bout : subset-sum vers Partition

Le problème **Partition** :

> **Instance** : une liste d'entiers positifs.
> **Question** : peut-on la couper en deux parts de même somme ?

**Réduire** subset-sum à Partition, c'est écrire une transformation qui change **toute** instance
de subset-sum en une instance de Partition **de même réponse**. Alors un algorithme pour Partition
résout aussi subset-sum : on transforme, on résout, on retraduit la solution.

La transformation classique ajoute deux nombres. Pour des poids de somme $\Sigma$ et une cible
$0 \le t \le \Sigma$ :

$$\text{poids} \;\longmapsto\; \text{poids} + [\,\Sigma + t,\; 2\Sigma - t\,]$$

La nouvelle liste a pour somme $4\Sigma$, donc chaque part doit valoir $2\Sigma$. Les deux nombres
ajoutés font ensemble $3\Sigma > 2\Sigma$ : ils sont forcément dans des parts **différentes**. La
part qui contient $2\Sigma - t$ doit donc compléter avec des poids d'origine de somme exactement
$t$ — c'est un certificat de subset-sum. Et réciproquement, un sous-ensemble de somme $t$ donne
une partition. Si $t > \Sigma$, la réponse est *non* : on renvoie une instance de Partition sans
solution, par exemple `[1]`.

Voici les quatre étapes, exécutées sur un exemple.

In [7]:
int[] ReduireSubsetVersPartition(int[] poids, int cible, out int marqueur)
{
    int sigma = poids.Sum();
    if (cible < 0 || cible > sigma) { marqueur = -1; return new[] { 1 }; }
    marqueur = poids.Length + 1;
    return poids.Concat(new[] { sigma+cible, 2*sigma-cible }).ToArray();
}
int[] ResoudrePartition(int[] valeurs)
{
    int somme = valeurs.Sum();
    if (somme % 2 != 0) return null;
    int cible = somme/2;
    var parent = new Dictionary<int,(int Avant,int Index)> { [0] = (-1,-1) };
    for (int i = 0; i < valeurs.Length; i++)
    {
        foreach (int s in parent.Keys.ToArray())
        {
            int suivant = s+valeurs[i];
            if (suivant <= cible && !parent.ContainsKey(suivant)) parent[suivant] = (s,i);
        }
    }
    if (!parent.ContainsKey(cible)) return null;
    var indices = new List<int>();
    for (int s = cible; s > 0; s = parent[s].Avant) indices.Add(parent[s].Index);
    return indices.ToArray();
}
int[] RetraduirePartition(int[] indicesPart, int[] valeurs, int nombreOriginaux, int marqueur)
{
    if (indicesPart == null) return null;
    bool utiliseMarqueur = indicesPart.Contains(marqueur);
    var cote = new HashSet<int>(indicesPart);
    var choix = new List<int>();
    for (int i = 0; i < nombreOriginaux; i++) if (cote.Contains(i) == utiliseMarqueur) choix.Add(i);
    int cible = valeurs.Take(nombreOriginaux).Where((_,i) => choix.Contains(i)).Sum();
    return choix.ToArray();
}
var exemplePoids = new[] { 3,34,4,12,5,2 }; int exempleCible = 9;
var partition = ReduireSubsetVersPartition(exemplePoids,exempleCible,out int indiceMarqueur);
var selection = ResoudrePartition(partition);
var certificat = RetraduirePartition(selection,partition,exemplePoids.Length,indiceMarqueur);
int sommeGauche = selection.Sum(i => partition[i]);
int sommeTotale = partition.Sum();
var verifReduction = VerifierSubset(exemplePoids,exempleCible,certificat);
if (selection == null || 2*sommeGauche != sommeTotale || !verifReduction.Accepte) throw new Exception("La réduction/rétraduction a échoué.");
Console.WriteLine($"Subset-sum : poids [{string.Join(", ",exemplePoids)}], cible {exempleCible}");
Console.WriteLine($"Partition : [{string.Join(", ",partition)}], {partition.Length} nombres, somme totale {sommeTotale}");
Console.WriteLine($"Certificat de moitié : indices [{string.Join(", ",selection)}], somme {sommeGauche}");
Console.WriteLine($"Certificat subset-sum retraduit : indices [{string.Join(", ",certificat)}], vérifié en {verifReduction.Operations} opérations.");

Subset-sum : poids [3, 34, 4, 12, 5, 2], cible 9


Partition : [3, 34, 4, 12, 5, 2, 69, 111], 8 nombres, somme totale 240


Certificat de moitié : indices [6, 4, 3, 1], somme 120


Certificat subset-sum retraduit : indices [0, 2, 5], vérifié en 3 opérations.


**Lecture.** La réduction ajoute les deux valeurs prévues et produit huit nombres dont la somme est **240**. Le solveur DP trouve une part de somme **120**; la part affichée exclut le marqueur (indice 7) et ses poids d’origine somment à 51. On prend donc son complément, qui contient le marqueur : les indices subset-sum `[0, 2, 5]`, de poids `[3, 4, 2]`, somment bien à la cible 9. Le vérificateur indépendant accepte le certificat en **3 opérations**.

In [8]:
var rngReduction = new Random(2026);
int oui = 0, non = 0, certificatsVerifies = 0;
for (int essai = 0; essai < 400; essai++)
{
    int n = rngReduction.Next(3,12);
    int[] poids = Enumerable.Range(0,n).Select(_ => rngReduction.Next(1,30)).ToArray();
    int cible = rngReduction.Next(0,poids.Sum()+10);
    var reduction = ReduireSubsetVersPartition(poids,cible,out int marqueur);
    var partie = ResoudrePartition(reduction);
    int[] cert = RetraduirePartition(partie,reduction,n,marqueur);
    bool oracle = SubsetBrut(poids,cible).Verdict;
    bool verdictReduction = partie != null;
    if (oracle != verdictReduction) throw new Exception($"Verdict divergent à l'instance {essai}.");
    if (oracle)
    {
        if (!VerifierSubset(poids,cible,cert).Accepte) throw new Exception($"Certificat retraduit invalide à l'instance {essai}.");
        certificatsVerifies++;
    }
    else if (cert != null) throw new Exception("Certificat inattendu dans un cas non.");
    if (oracle) oui++; else non++;
}
if (oui+non != 400 || certificatsVerifies != oui) throw new Exception("Couverture de l'expérience incomplète.");
Console.WriteLine($"400 instances (seed .NET 2026) : {oui} oui, {non} non.");
Console.WriteLine($"Transformation → Partition DP → retraduction : {oui} certificats retraduits et revérifiés ; aucun écart de verdict.");

400 instances (seed .NET 2026) : 227 oui, 173 non.


Transformation → Partition DP → retraduction : 227 certificats retraduits et revérifiés ; aucun écart de verdict.


**Lecture.** La chaîne complète concorde avec le solveur direct sur les **400 instances** tirées avec le seed .NET **2026** : **227 oui**, **173 non**, et les **227 certificats** retraduits sont revérifiés. Le générateur .NET n'a pas la même suite que NumPy; la taille de l'échantillon et le seed sont préservés. Il s'agit d'un contrôle expérimental, tandis que la preuve de correction vient de l'argument de réduction.

### Exercice 2 — Réduire Partition vers subset-sum

Une partition en deux groupes de même somme se transforme en subset-sum. Implémentez `ReduirePartitionVersSubset(valeurs)` et renvoyez `(poids, cible)` ou `null` tant que l'exercice est incomplet. La somme impaire doit donner une instance impossible. La garde vérifie une instance paire `[1, 5, 6]`, une instance impaire `[1, 2]`, puis 200 listes générées avec seed 11, de tailles 2 à 9. La réponse subset-sum doit correspondre au solveur exact de Partition sur chaque entrée; les dimensions, poids positifs et cible sont également contrôlés.

In [9]:
(int[] Poids,int Cible)? ReduirePartitionVersSubset(int[] valeurs)
{
    // TODO étudiant : retourner une instance de subset-sum équivalente.
    return null;
}
bool TestReductionInverse(int[] valeurs,bool attendu)
{
    var resultat = ReduirePartitionVersSubset(valeurs);
    if (!resultat.HasValue) return false;
    var (poids,cible) = resultat.Value;
    if (poids == null || poids.Length == 0 || poids.Length > valeurs.Length+1 || cible < 0 || poids.Any(p => p <= 0) || cible > poids.Sum()) return false;
    return SubsetDp(poids,cible).Verdict == attendu;
}
var fixePair = TestReductionInverse(new[] { 1,5,6 }, true);
var fixeImpair = TestReductionInverse(new[] { 1,2 }, false);
var rngInverse = new Random(11);
int conformes = 0, ouiPartition = 0, nonPartition = 0, echecs = 0;
for (int essai = 0; essai < 200; essai++)
{
    int n = rngInverse.Next(2,10);
    int[] valeurs = Enumerable.Range(0,n).Select(_ => rngInverse.Next(1,25)).ToArray();
    bool attendu = ResoudrePartition(valeurs) != null;
    if (!TestReductionInverse(valeurs,attendu)) echecs++;
    if (attendu) ouiPartition++; else nonPartition++;
    conformes++;
}
if (fixePair && fixeImpair && conformes == 200 && ouiPartition > 0 && nonPartition > 0 && echecs == 0)
    Console.WriteLine($"Réduction inverse validée : fixtures paire/impaire et {conformes} instances seed 11 ({ouiPartition} oui, {nonPartition} non).");
else Console.WriteLine($"Exercice non résolu ou incorrect : implémentez ReduirePartitionVersSubset; fixtures paire/impaire={fixePair}/{fixeImpair}, {echecs}/{conformes} cas en désaccord. Aucun résultat n'est certifié.");

Exercice non résolu ou incorrect : implémentez ReduirePartitionVersSubset; fixtures paire/impaire=False/False, 200/200 cas en désaccord. Aucun résultat n'est certifié.


**Lecture.** Avec le stub livré, les fixtures sont refusées et aucun résultat n'est certifié; une implémentation doit aussi réussir les 200 cas oui/non contrôlés. Une réduction correcte envoie une liste de somme paire vers les mêmes poids et une cible égale à la moitié; une somme impaire devient un cas impossible.

## 5. Pourquoi l'exponentielle est le mur de l'ingénieur

Revenons à la force brute sur subset-sum. Mesurons combien d'opérations par seconde cette machine
exécute, puis extrapolons le temps nécessaire, dans le pire cas ($n \cdot 2^n$ opérations), pour
des tailles plus grandes — sur cette machine et sur une machine **mille fois** plus rapide.

In [10]:
string Lisible(double secondes)
{
    if (secondes < 1) return $"{secondes*1000:F2} ms";
    if (secondes < 60) return $"{secondes:F2} s";
    if (secondes < 3600) return $"{secondes/60:F2} min";
    if (secondes < 86400) return $"{secondes/3600:F2} h";
    if (secondes < 365.25*86400) return $"{secondes/86400:F2} jours";
    return $"{(secondes/(365.25*86400)).ToString("N2",CultureInfo.InvariantCulture).Replace(',', ' ')} ans";
}
int nMesure = 18;
var chrono = Stopwatch.StartNew();
var mesure = SubsetBrut(Enumerable.Repeat(2,nMesure).ToArray(),2*nMesure-1);
chrono.Stop();
long operationsParDefaut = (long)nMesure*(1L<<nMesure);
if (mesure.Verdict || mesure.Operations != operationsParDefaut) throw new Exception("La mesure réelle n'a pas parcouru l'espace prévu.");
double duree = chrono.Elapsed.TotalSeconds;
double vitesse = mesure.Operations/duree;
double vitesseMille = vitesse*1000;
Console.WriteLine($"Mesure réelle : n={nMesure}, {mesure.Operations:N0} opérations en {Lisible(duree)} ({vitesse:N0} opérations/s).");
Console.WriteLine(" n | opérations exactes | cette machine | machine ×1 000 | vérifier un certificat");
Console.WriteLine(new string('-',86));
foreach (int n in new[] { 20,30,40,50,60,80 })
{
    BigInteger operations = (BigInteger)n * (BigInteger.One << n);
    double secondes = (double)operations/vitesse;
    double secondesMille = (double)operations/vitesseMille;
    Console.WriteLine($"{n,2} | {operations.ToString("N0",CultureInfo.InvariantCulture),19} | {Lisible(secondes),13} | {Lisible(secondesMille),13} | {n} opérations");
}
int FrontiereUnJour(double debit)
{
    int n = 0;
    while ((double)((BigInteger)(n+1)*(BigInteger.One<<(n+1)))/debit <= 86400) n++;
    return n;
}
int nUnJour = FrontiereUnJour(vitesse), nUnJourMille = FrontiereUnJour(vitesseMille);
Console.WriteLine($"Seuil d'une journée : n={nUnJour} ici, n={nUnJourMille} à ×1 000 (gain de {nUnJourMille-nUnJour} poids).");

Mesure réelle : n=18, 4,718,592 opérations en 64.20 ms (73,504,042 opérations/s).


 n | opérations exactes | cette machine | machine ×1 000 | vérifier un certificat


--------------------------------------------------------------------------------------


20 |          20,971,520 |     285.31 ms |       0.29 ms | 20 opérations


30 |      32,212,254,720 |      7.30 min |     438.24 ms | 30 opérations


40 |  43,980,465,111,040 |    6.93 jours |      9.97 min | 40 opérations


50 | 56,294,995,342,131,200 |     24.27 ans |    8.86 jours | 50 opérations


60 | 69,175,290,276,410,818,560 | 29 821.93 ans |     29.82 ans | 60 opérations


80 | 96,714,065,569,170,333,976,494,080 | 41 694 085 165.32 ans | 41 694 085.17 ans | 80 opérations


Seuil d'une journée : n=37 ici, n=46 à ×1 000 (gain de 9 poids).


**Lecture.** Le parcours complet à $n=18$ mesure réellement **4 718 592 opérations** ; la première ligne donne le débit propre à cette exécution. Les durées des six tailles suivantes sont calculées en divisant leur nombre exact d’opérations par ce débit : ce sont des **extrapolations**, pas des mesures de durées aussi longues. Multiplier le débit par mille ne repousse ici le seuil d’une journée que de **9 poids** (les deux seuils exacts sont affichés). Le débit et les chronos varient avec la machine ; la croissance $n\cdot2^n$ reste la même.

### Exercice 3 — Reconstruire avec un oracle oui/non

L'oracle `existe(poids, cible)` répond seulement si un sous-ensemble existe. Implémentez `TrouverAvecOracle(poids, cible, existe)` pour reconstruire un certificat, en faisant au plus `n+1` appels. Le contrôle utilise une instance oui `[3, 34, 4, 12, 5, 2]`, cible 9, et une instance non `[2, 2]`, cible 3. Il vérifie que les indices sont distincts et valides, que le certificat somme à la cible, que le compteur correspond aux appels réels et que la borne `n+1` tient dans les deux cas.

In [11]:
(bool Trouve,int[] Indices,int Appels)? TrouverAvecOracle(int[] poids,int cible,Func<int[],int,bool> existe)
{
    // TODO étudiant : reconstruire un certificat avec au plus n+1 appels à l'oracle.
    return null;
}
var poidsOracle = new[] { 3,34,4,12,5,2 }; int cibleOracle = 9;
int appelsOui = 0;
bool ExisteOui(int[] p,int t) { appelsOui++; return SubsetDp(p,t).Verdict; }
var trouve = TrouverAvecOracle(poidsOracle,cibleOracle,ExisteOui);
bool valideOui = trouve.HasValue && trouve.Value.Trouve && trouve.Value.Indices != null &&
    trouve.Value.Appels == appelsOui && appelsOui >= 1 && appelsOui <= poidsOracle.Length+1 &&
    trouve.Value.Indices.Distinct().Count() == trouve.Value.Indices.Length &&
    trouve.Value.Indices.All(i => i >= 0 && i < poidsOracle.Length) &&
    VerifierSubset(poidsOracle,cibleOracle,trouve.Value.Indices).Accepte;
int appelsNon = 0;
bool ExisteNon(int[] p,int t) { appelsNon++; return SubsetDp(p,t).Verdict; }
var impossible = TrouverAvecOracle(new[] { 2,2 },3,ExisteNon);
bool valideNon = impossible.HasValue && !impossible.Value.Trouve && impossible.Value.Indices != null &&
    impossible.Value.Indices.Length == 0 && impossible.Value.Appels == appelsNon && appelsNon >= 1 && appelsNon <= 3;
if (valideOui && valideNon)
    Console.WriteLine($"Oracle : certificat {string.Join(",",trouve.Value.Indices)} validé en {appelsOui}/{poidsOracle.Length+1} requêtes; cas impossible rejeté en {appelsNon}/3.");
else Console.WriteLine("Exercice non résolu : implémentez TrouverAvecOracle; aucun résultat stub n'est affiché comme valide.");

Exercice non résolu : implémentez TrouverAvecOracle; aucun résultat stub n'est affiché comme valide.


**Lecture.** Le stub n'invente ni certificat ni nombre d'appels. Une solution doit vérifier le résultat avec le vérificateur subset-sum et compter chaque appel réel à l'oracle; la limite $n+1$ comprend le test initial de faisabilité.

## Conclusion

| Problème | Vérifier un certificat | Trouver une solution (algorithmes étudiés ici) |
|---|---|---|
| Sudoku $m^2 \times m^2$ | au plus 4 lectures par case (**mesuré** : 324 pour $m = 3$) | exponentiel dans le pire cas (backtracking **mesuré** jusqu'à des centaines de milliers d'appels) |
| subset-sum | une addition par poids choisi (**mesuré**) | force brute $n \cdot 2^n$ (**mesuré**) ; programmation dynamique pseudo-polynomiale (**mesuré**) |
| Partition | une addition par élément | aussi dur que subset-sum : réduction **exécutée** sur 400 instances |

Ce que ce notebook a établi :

1. **NP** est la classe des problèmes dont les solutions se **vérifient** vite ; **P** celle des
   problèmes qui se **résolvent** vite. P ⊆ NP, et l'égalité est la grande question ouverte.
2. Une **réduction** compare deux problèmes en transformant les instances de l'un en instances de
   l'autre ; elle se programme, s'exécute et se teste.
3. Le coût se mesure en fonction de la **taille écrite** de l'entrée : la programmation dynamique
   de subset-sum n'est polynomiale qu'en apparence.

Le notebook suivant explique l'idée de hiérarchie temporelle : donner plus de temps peut permettre
de résoudre de nouveaux problèmes, mais il faut le **prouver**, pas seulement comparer deux
programmes. L'approfondissement 03b conserve le théorème de Hartmanis et Stearns (1965).

**Suite** : [Complexity-03 — Plus de temps, plus de problèmes >>](Complexity-03-HierarchieTemporelle-Python.ipynb)

## À retenir

1. **Trouver varie, vérifier est borné.** Les quatre Sudoku demandent 49, 19 023, 335 638 et 228 215 appels; le vérificateur lit toujours 324 cases, et rejette le faux certificat après 180 lectures.
2. **Sans solution, la force brute fait exactement $n \cdot 2^n$.** Les cas $n=10$ et $n=18$ comptent 10 240 et 4 718 592 opérations; les ratios observés pour +2 poids suivent la formule.
3. **Le pseudo-polynomial suit la valeur des sommes.** Sur cette entrée seed 7, multiplier les poids par 10 000 garde 5 264 opérations brutes, tandis que la DP augmente de 2 762 à 27 480 014 transitions; la longueur totale écrite passe de 28 à 88 chiffres.
4. **La réduction est testée de bout en bout.** Sur 400 instances seed .NET 2026, les verdicts concordent : 227 oui, 173 non, et les 227 certificats retraduits passent le vérificateur. Un exemple produit une part de 120 et le certificat `[0, 2, 5]`, accepté en 3 opérations.
5. **Le mur est quantifié par extrapolation.** La mesure à $n=18$ porte sur **4 718 592 opérations**. Les durées des tailles supérieures utilisent le débit affiché, sans prétendre mesurer ces longues exécutions. Une machine mille fois plus rapide ne décale ici le seuil d’une journée que de **9 poids** ; les nombres exacts d’opérations restent ceux de la table.

## Vérifiez votre compréhension

1. Les quatre puzzles demandent respectivement 49, 19 023, 335 638 et 228 215 appels, tandis que le vérificateur lit toujours 324 cases. Qu'est-ce qui varie, et pourquoi le coût de vérification reste-t-il borné ?
2. Le certificat falsifié est rejeté après 180 lectures. Qu'est-ce qui distingue vérifier un certificat de lui faire confiance ?
3. La force brute sans solution passe de 10 240 opérations à $n=10$ à 4 718 592 à $n=18$. Pourquoi ajouter deux poids augmente-t-il le travail d'environ un facteur quatre ?
4. Sur l'expérience seed 7, multiplier les poids par 10 000 fait passer la DP de 2 762 à 27 480 014 transitions, alors que la longueur totale écrite des poids et de la cible passe de 28 à 88 chiffres. Qu'est-ce que cela dit sur la taille de l'entrée ?
5. En utilisant le débit affiché par la cellule de mesure, comment projeter le temps pour $n=30$ ? Pourquoi une machine mille fois plus rapide ne réduit-elle pas l'exposant de moitié ?

<details><summary>Réponses</summary>

1. Le nombre d'appels dépend du chemin de recherche et de la difficulté de chaque grille. Le vérificateur relit les 81 indices puis les 27 unités : son nombre maximal de lectures dépend uniquement de la taille fixée de la grille.
2. Le vérificateur compare les indices et recalcule les contraintes de chaque ligne, colonne et bloc; il refuse une proposition qui les viole.
3. $n \cdot 2^n$ est multiplié par $4(n+2)/n$ quand $n$ augmente de deux, soit un facteur proche de quatre pour ces tailles.
4. La DP indexe ses états par valeur de somme; l'écriture binaire des nombres ne grandit que de quelques chiffres alors que la table s'agrandit en proportion de la cible. Le coût est pseudo-polynomial.
5. Diviser $30\cdot2^{30} = 32\,212\,254\,720$ opérations par le débit affiché donne une projection en secondes, et non une mesure de cette exécution. Une accélération constante ne change que le multiplicateur du temps. Pour compenser ×1 000 dans $2^n$, il suffit d'environ $\log_2(1000) \approx 10$ poids de moins; l'exposant reste exponentiel.

</details>

## Pour aller plus loin

- **Carnet suivant** : [Complexity-03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb) — l'idée de hiérarchie temporelle : donner plus de temps peut ouvrir de nouveaux problèmes, mais il faut le **prouver**, pas seulement comparer deux programmes.
- **Dans la même série** : [Complexity-02b — le seuil du 3-SAT](Complexity-02b-SAT-PhaseTransition-Python.ipynb) — la densité comme variable de contrôle, à $n$ fixé ; [Complexity-02c — le seuil sans mur](Complexity-02c-XOR-SAT-Phase-Transition-Python.ipynb) — un contre-exemple propre, où la transition de satisfaisabilité ne s'accompagne d'aucun mur de coût.
- **Carnet voisin d'une autre série** : [CSP-2 — Consistency](../Search/Part2-CSP/CSP-2-Consistency.ipynb) — la satisfiabilité par propagation de contraintes, mesurée dans une autre série.
- **Lectures** : Impagliazzo et Paturi (2001), *On the Complexity of k-SAT*, J. Comput. Syst. Sci. 62-2 (copie archivée au gisement partagé, dossier `Bibliographie IA/Automata/`) ; Dechter, *Constraint Processing* (2003) — la famille des solveurs par retour arrière et la façon de mesurer leur coût (dossier `Bibliographie IA/Constraint Programming/`).